# Backtesting

Imports

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import statsmodels.api as sm

Matplotlib is building the font cache; this may take a moment.


### Step 1: Load cointegration and price data

In [2]:
data_folder_path = Path("data")
print(data_folder_path.exists())

True


In [3]:
bh_corrected_df = pd.read_parquet(data_folder_path / "bh_corrected_results.parquet")
bh_sig_results_df = bh_corrected_df[bh_corrected_df["bh_reject"]]
sig_results_df = pd.read_parquet(data_folder_path / "sig_results.parquet")

In [4]:
print(sig_results_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    STT      GS           0.753596 -3.362022  0.046747   
                       MS           0.739525 -4.778833  0.000411   
                       LNC          0.776851 -3.481309  0.034089   
              COP      EOG          0.781141 -3.454777  0.036620   
              KIM      O            0.762781 -4.339056  0.002224   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    STT      GS            -3.918346      -3.348304       -3.052893   
                       MS            -3.918346      -3.348304       -3.052893   
                       LNC           -3.918346      -3.348304       -3.052893   
              COP      EOG           -3.918346      -3.348304       -3.052893   
              KIM      O             

In [5]:
print(bh_corrected_df.head())

                                 correlation   coint_t   p_value  \
formation_end ticker_0 ticker_1                                    
2016-01-01    SWKS     AVGO         0.705393 -0.019827  0.985429   
              BAC      STT          0.730833 -2.431513  0.310079   
                       HBAN         0.767496 -2.654967  0.216192   
                       PFG          0.710855 -2.270825  0.388197   
                       TFC          0.756456 -2.563165  0.251944   

                                 critical_1pct  critical_5pct  critical_10pct  \
formation_end ticker_0 ticker_1                                                 
2016-01-01    SWKS     AVGO          -3.918346      -3.348304       -3.052893   
              BAC      STT           -3.918346      -3.348304       -3.052893   
                       HBAN          -3.918346      -3.348304       -3.052893   
                       PFG           -3.918346      -3.348304       -3.052893   
                       TFC           

In [6]:
historical_ohlcv_df = pd.read_parquet(data_folder_path / "historical_ohlcv_df.parquet")

In [7]:
print(historical_ohlcv_df.head())

Ticker     SOLV                       KVUE                        ...  \
Price      Open High Low Close Volume Open High Low Close Volume  ...   
Date                                                              ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   

Ticker              R                                                 EOG  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2014-01-02  51.244708  51.384891  50.683965  50.915272  453500  59.113503   
2014-01-03  51.055471  51.335842  50.838185  51.020428  303500  58.822462   
2014-01-06  51.265752  51.4479

### Step 2: Estimate relationship parameters for all pairs

In [8]:
close_prices = historical_ohlcv_df.xs("Close", level="Price", axis=1)
print(close_prices.head())

Ticker      SOLV  KVUE  SNI  LLL  WBA  EXE       CBRE         MTD  ADT  \
Date                                                                     
2014-01-02   NaN   NaN  NaN  NaN  NaN  NaN  26.340000  241.529999  NaN   
2014-01-03   NaN   NaN  NaN  NaN  NaN  NaN  26.400000  242.710007  NaN   
2014-01-06   NaN   NaN  NaN  NaN  NaN  NaN  26.250000  242.419998  NaN   
2014-01-07   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  248.360001  NaN   
2014-01-08   NaN   NaN  NaN  NaN  NaN  NaN  26.190001  249.669998  NaN   

Ticker            IQV  ...        GIS   AMD        CAG       MKTX       SCHW  \
Date                   ...                                                     
2014-01-02  45.799999  ...  31.121460  3.95  16.424904  57.739132  22.172007   
2014-01-03  46.290001  ...  31.033251  4.00  16.385775  59.292953  22.240696   
2014-01-06  46.029999  ...  31.083658  4.13  16.307476  57.483101  22.163406   
2014-01-07  46.790001  ...  31.404943  4.18  16.586372  58.030479  21.931561   
2

Create a function to estimate params for a pair

In [9]:
def estimate_params(ticker_0, ticker_1, formation_start, formation_end):
    pair_prices = close_prices.loc[
        formation_start:(formation_end - pd.DateOffset(days=1)),
        [ticker_0, ticker_1]
    ].dropna()

    y = pair_prices[ticker_0]
    x = pair_prices[ticker_1]

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params["const"]
    beta = model.params[ticker_1]

    spread = y - (alpha + beta * x)

    return alpha, beta, spread.mean(), spread.std()

In [10]:
print(estimate_params("AAPL", "MSFT", pd.Timestamp("2020-01-01"), pd.Timestamp("2020-12-31")))

(np.float64(-70.92463045076722), np.float64(0.8908157857894756), np.float64(1.1752151280349595e-13), np.float64(7.600241702353187))


Create a function to estimate params for all pairs in a window and store in a dataframe

In [11]:
def estimate_params_for_window(pairs, formation_start, formation_end):
    results = []

    for ticker_0, ticker_1 in pairs:
        alpha, beta, spread_mean, spread_std = estimate_params(ticker_0, ticker_1, formation_start, formation_end)
        results.append({
            "formation_end": formation_end,
            "ticker_0": ticker_0,
            "ticker_1": ticker_1,
            "alpha": alpha,
            "beta": beta,
            "spread_mean": spread_mean,
            "spread_std": spread_std
        })
    return pd.DataFrame(results).set_index(["formation_end", "ticker_0", "ticker_1"])

Create a function to estimate params for all windows in a dataframe of cointegration data. We will call this function twice: once on the raw p-value threshold data and once on the BH-corrected data.

In [12]:
def estimate_params_for_all_windows(coint_df, window_size=504):
    results = []

    formation_ends = coint_df.index.get_level_values("formation_end").unique()

    for formation_end in formation_ends:
        end_idx = close_prices.index.searchsorted(formation_end, side="left") - 1
        formation_start = close_prices.index[end_idx - (window_size - 1)]

        pairs = coint_df.loc[formation_end].index.tolist()

        window_params_df = estimate_params_for_window(
            pairs,
            formation_start,
            formation_end
        )

        results.append(window_params_df)

    return pd.concat(results, ignore_index=False)

Estimate params for both sig_results_df and bh_corrected_df

In [13]:
sig_window_params_df = estimate_params_for_all_windows(sig_results_df)
bh_window_params_df = estimate_params_for_all_windows(bh_sig_results_df)

In [14]:
sig_window_params_df.head()

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    STT      GS        15.529639  0.254871  7.330996e-16    1.591031
                       MS        15.050690  1.496055  8.049998e-15    1.330992
                       LNC        9.530159  1.223589  4.466269e-14    1.630143
              COP      EOG       -8.118959  0.813194  1.084424e-13    2.007876
              KIM      O          0.943563  0.495747 -4.854670e-14    0.274850

In [15]:
bh_window_params_df

alpha      beta   spread_mean  spread_std
formation_end ticker_0 ticker_1                                               
2016-01-01    XEL      CMS        3.514142  0.847090  5.116190e-14    0.228813
2016-02-01    XEL      CMS        3.464461  0.849943  4.366172e-14    0.236661
2016-03-01    STT      MS        10.544876  1.664574 -8.238912e-14    1.125561
              XEL      CMS        3.481574  0.849849 -7.390913e-14    0.234388
2016-04-01    STT      MS        11.365390  1.633517 -6.541505e-14    1.122996
...                                    ...       ...           ...         ...
2025-11-01    PPL      NI         8.675948  0.647139  3.207311e-15    0.611207
              CPT      ESS        3.367900  0.392093 -2.774218e-13    2.017918
              RF       KEY       -1.634339  1.484203  2.112596e-14    0.445216
2025-12-01    CPT      ESS        2.656230  0.394576 -7.993606e-14    2.043802
              RF       KEY       -2.206338  1.521070  1.817241e-14    0.419373

[406 rows x 4 columns]

### Simulate trading using formation window data

Calculate z-score from 2 prices

In [16]:
def calculate_z_score(price_0, price_1, alpha, beta, spread_mean, spread_std):
    spread = price_0 - (alpha + beta * price_1) # y is ticker_0, x is ticker_1 according to our earlier code
    return (spread - spread_mean) / spread_std

In [21]:
TRADE_THRESH = 2 # only trade if absolute value of z-score is > 2
SELL_THRESH = 0.5 # we will sell our current portfolio if absolute value of z score < 0.5
TRADE_NOMINAL = 10000 # spend $10k per trade
TRANSACTION_COST_RATE = 0.0005 # 5 bps per trade

Function to open a position

In [18]:
open_positions = {}
trade_history = []

def open_position(ticker_0, ticker_1, formation_end, direction, signal_date, entry_date, entry_z, alpha, beta, q0, q1, entry_price_0, entry_price_1):
    pair = (ticker_0, ticker_1)

    if pair in open_positions:
        raise ValueError(f"Position for pair {pair} is already open.")

    entry_gross_exposure = abs(q0 * entry_price_0) + abs(q1 * entry_price_1)
    entry_cost = TRANSACTION_COST_RATE * entry_gross_exposure

    open_positions[pair] = {
        "formation_end": formation_end,
        "direction": direction,
        "signal_date": signal_date,
        "entry_date": entry_date,
        "entry_z": entry_z,
        "alpha": alpha,
        "beta": beta,
        "q0": q0,
        "q1": q1,
        "entry_price_0": entry_price_0,
        "entry_price_1": entry_price_1,
        "entry_gross_exposure": entry_gross_exposure,
        "entry_cost": entry_cost
    }

Function to close a position

In [19]:
def close_position(ticker_0, ticker_1, exit_signal_date, exit_date, exit_z, exit_price_0, exit_price_1, exit_reason):
    pair = (ticker_0, ticker_1)

    if pair not in open_positions:
        raise ValueError(f"Position for pair {pair} is not open.")

    position = open_positions[pair]

    q0 = position["q0"]
    q1 = position["q1"]

    gross_pnl = q0 * (exit_price_0 - position["entry_price_0"]) + q1 * (exit_price_1 - position["entry_price_1"])

    exit_gross_exposure = abs(q0 * exit_price_0) + abs(q1 * exit_price_1)
    exit_cost = TRANSACTION_COST_RATE * exit_gross_exposure
    total_cost = position["entry_cost"] + exit_cost
    net_pnl = gross_pnl - total_cost
    trade_return = net_pnl / position["entry_gross_exposure"]
    holding_days = (exit_date - position["entry_date"]).days

    trade_history.append({
        "ticker_0": ticker_0,
        "ticker_1": ticker_1,
        "formation_end": position["formation_end"],
        "direction": position["direction"],
        "entry_signal_date": position["signal_date"],
        "entry_date": position["entry_date"],
        "exit_signal_date": exit_signal_date,
        "exit_date": exit_date,
        "entry_z": position["entry_z"],
        "exit_z": exit_z,
        "alpha": position["alpha"],
        "beta": position["beta"],
        "q0": q0,
        "q1": q1,
        "entry_price_0": position["entry_price_0"],
        "entry_price_1": position["entry_price_1"],
        "exit_price_0": exit_price_0,
        "exit_price_1": exit_price_1,
        "entry_gross_exposure": position["entry_gross_exposure"],
        "exit_gross_exposure": exit_gross_exposure,
        "gross_pnl": gross_pnl,
        "entry_cost": position["entry_cost"],
        "exit_cost": exit_cost,
        "total_cost": total_cost,
        "net_pnl": net_pnl,
        "trade_return": trade_return,
        "holding_days": holding_days,
        "exit_reason": exit_reason
    })

    del open_positions[pair]

Function to trade within a window for a given df

In [ ]:
def simulate_trading_window(params_df, formation_end):
    start_date = formation_end
    end_date = formation_end + pd.DateOffset(months=1)

    trading_ohlcv_df = historical_ohlcv_df.loc[
        (historical_ohlcv_df.index >= start_date) &
        (historical_ohlcv_df.index <= end_date)
    ]

    trading_close_df = trading_ohlcv_df.xs('Close', level='Price', axis=1).iloc[:-1] # used for signals
    trading_open_df = trading_ohlcv_df.xs('Open', level='Price', axis=1).iloc[1:] # used for returns

    formation_params = params_df.xs(formation_end, level="formation_end")
    pairs = formation_params.index.tolist()

    #print(trading_close_df)

    for curr_date in trading_close_df.index:
        daily_close_prices = trading_close_df.loc[curr_date]
        open_date = curr_date + pd.DateOffset(days=1)
        next_open_prices = trading_open_df.loc[open_date]
        for ticker_0, ticker_1 in pairs:
            # get prices for each ticker
            close_price_0 = daily_close_prices[ticker_0]
            close_price_1 = daily_close_prices[ticker_1]

            # get params & z-score
            curr_pair_params = formation_params.loc[(ticker_0, ticker_1)] # extracts a Series since pairs are unique
            alpha = curr_pair_params["alpha"]
            beta = curr_pair_params["beta"]
            spread_mean = curr_pair_params["spread_mean"]
            spread_std = curr_pair_params["spread_std"]
            z_score = calculate_z_score(close_price_0, close_price_1, alpha, beta, spread_mean, spread_std)

            if (ticker_0, ticker_1) in open_positions:
                if (curr_date == trading_close_df.index[-1]):
                    close_position(ticker_0, ticker_1, curr_date, open_date, z_score, next_open_prices[ticker_0], next_open_prices[ticker_1], "window_end")

                if (abs(z_score) <= 0.5):
                    close_position(ticker_0, ticker_1, curr_date, open_date, z_score, next_open_prices[ticker_0], next_open_prices[ticker_1], "mean_reversion")

                continue


            if (abs(z_score) >= 2):
                entry_price_0 = next_open_prices[ticker_0]
                entry_price_1 = next_open_prices[ticker_1]

                scale = TRADE_NOMINAL / (entry_price_0 + abs(beta) * entry_price_1)

                direction = z_score / abs(z_score)
                q0 = direction * scale # shares of ticker_1
                q1 = -direction * beta * scale # shares of ticker_2
                # execute trade for this pair
                open_position(ticker_0, ticker_1, formation_end, direction, curr_date, open_date, z_score, alpha, beta, q0, q1, entry_price_0, entry_price_1)

        curr_date = open_date

    # print(pairs)
simulate_trading_window(sig_window_params_df, pd.Timestamp("2016-01-01"))